# Phase 2N - Removing the near-duplicates directly

**Run on:** Kaggle or Colab, free T4. ~60 min for 30 training runs.

---

### Why this exists

Every measurement of the leakage effect so far compares an **image-level**
partition against a **grouped** one. That comparison changes more than one thing:
it separates near-duplicates, but it also imposes a blocked structure and shifts
the class prevalence of the test folds. The random-group control addressed the
blocking. It did not address the prevalence shift, and it did not touch
near-duplicates directly.

This notebook tests the mechanism head-on, with no grouping anywhere.

| arm | training set | test set |
|---|---|---|
| image-level | full | fold's test images |
| **dedup** | **near-duplicates of test images removed** | **identical** |
| random-removal | same number removed, chosen at random | identical |

**The test folds are byte-identical across all three arms.** Accuracy is
therefore directly comparable, with no partition change to argue about.

### What each outcome means, fixed before the run

- **dedup drops, random-removal does not:** near-duplicate leakage is confirmed
  directly. The paper's title is earned.
- **both drop by a similar amount:** the effect is training-set size, not
  duplication. The central claim fails.
- **neither drops:** the image-level score was never inflated by duplication,
  and the grouped-protocol gap must be explained some other way.

We report whichever occurs. The interpretation above is fixed here, in the
notebook, before any result exists.

### The one design choice worth stating

Duplicates are removed from **training**, not test. Removing them from test
would change the test set between arms and make the accuracies incomparable,
which is the mistake that would quietly invalidate the whole experiment.

## 1. Environment and data

In [ ]:
import subprocess, sys
for pkg in ["opencv-python-headless", "tabulate", "kagglehub", "scipy"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", pkg],
                   check=False)

import os, json, time, shutil, random, gc
import numpy as np
import pandas as pd
import cv2
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (precision_recall_fscore_support, accuracy_score,
                             balanced_accuracy_score)

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
print("GPUs:", tf.config.list_physical_devices("GPU"))

IN_KAGGLE = os.path.exists("/kaggle/working")
WORK    = "/kaggle/working" if IN_KAGGLE else "/content"
SCRATCH = "/kaggle/temp"    if IN_KAGGLE else "/content"
try:
    os.makedirs(SCRATCH, exist_ok=True)
except OSError:
    SCRATCH = "/tmp"; os.makedirs(SCRATCH, exist_ok=True)

RESULTS_DIR = f"{WORK}/fyp_phase2n_results"
os.makedirs(RESULTS_DIR, exist_ok=True)
N_FOLDS, DUP_THRESHOLD = 5, 0.98
RESULTS = {"seed": SEED, "n_folds": N_FOLDS, "dup_threshold": DUP_THRESHOLD}

def save_json():
    with open(f"{RESULTS_DIR}/results.json", "w") as f:
        json.dump(RESULTS, f, indent=2, default=float)
print("outputs ->", RESULTS_DIR)

In [ ]:
FOLDERS = {"Benign": "Bengin cases", "Malignant": "Malignant cases",
           "Normal": "Normal cases"}

DATA_ROOT = None
if os.path.isdir("/kaggle/input"):
    hits = [d for d, _, _ in os.walk("/kaggle/input")
            if os.path.basename(d) == "Malignant cases"]
    if hits:
        DATA_ROOT = os.path.dirname(hits[0])
        print("using the attached dataset:", DATA_ROOT)
if DATA_ROOT is None:
    import kagglehub
    DL = kagglehub.dataset_download("hamdallak/the-iqothnccd-lung-cancer-dataset")
    cands = [d for d, _, _ in os.walk(DL) if os.path.basename(d) == "Malignant cases"]
    DATA_ROOT = os.path.dirname(cands[0])
    print("downloaded via kagglehub:", DATA_ROOT)

SPLIT_URL = ("https://raw.githubusercontent.com/haseebkhan9081/"
             "iqothnccd-leakage-audit/main/split_seed42.csv")
subprocess.run(["wget", "-q", "-O", f"{SCRATCH}/split_seed42.csv", SPLIT_URL],
               check=True)
df = pd.read_csv(f"{SCRATCH}/split_seed42.csv")
print("columns:", list(df.columns))
df["path"] = [os.path.join(DATA_ROOT, FOLDERS[l], f)
              for l, f in zip(df["label"], df["file"])]
assert all(os.path.exists(p) for p in df["path"]), "image paths do not resolve"
y_all = df["y"].to_numpy()
print("images:", len(df))
RESULTS["data"] = {"n_images": int(len(df))}
save_json()

## 2. The similarity matrix

The same 64x64 mean-centred, L2-normalised thumbnail representation used
throughout the paper, so "near-duplicate" means here exactly what it means
elsewhere.

In [ ]:
thumbs = []
for p in tqdm(df["path"], desc="thumbnails"):
    g = cv2.cvtColor(cv2.imread(p), cv2.COLOR_BGR2GRAY)
    g = cv2.resize(g, (64, 64), interpolation=cv2.INTER_AREA).astype(np.float32).ravel()
    g -= g.mean()
    n = np.linalg.norm(g)
    thumbs.append(g / n if n > 0 else g)
T = np.stack(thumbs)
S = T @ T.T
np.fill_diagonal(S, 0.0)

n_pairs = int((np.triu(S, 1) >= DUP_THRESHOLD).sum())
print(f"pairs with cosine >= {DUP_THRESHOLD}: {n_pairs}")
RESULTS["near_duplicate_pairs_total"] = n_pairs
save_json()

## 3. The three arms

For each image-level fold, find the training images that are near-duplicates of
some test image. Remove them. Then remove the same *number* of training images
at random, as a size control.

In [ ]:
folds = list(StratifiedKFold(n_splits=N_FOLDS, shuffle=True,
                             random_state=SEED).split(df, y_all))
rng = np.random.default_rng(SEED)

ARMS = {}
fold_stats = []
for k, (tr, te) in enumerate(folds):
    # a training image is contaminated if it is >= threshold similar to ANY test image
    contaminated = (S[np.ix_(tr, te)] >= DUP_THRESHOLD).any(axis=1)
    tr_dedup = tr[~contaminated]
    n_removed = int(contaminated.sum())
    tr_random = rng.choice(tr, size=len(tr) - n_removed, replace=False)

    ARMS.setdefault("image-level", []).append((tr, te))
    ARMS.setdefault("dedup", []).append((tr_dedup, te))
    ARMS.setdefault("random-removal", []).append((np.sort(tr_random), te))

    # the removed images, by class, so any class imbalance introduced is visible
    by_class = np.bincount(y_all[tr][contaminated], minlength=3).tolist()
    fold_stats.append({"fold": k, "n_train": int(len(tr)), "n_test": int(len(te)),
                       "n_removed": n_removed,
                       "pct_removed": round(100 * n_removed / len(tr), 2),
                       "removed_by_class": by_class})
    print(f"fold {k}: {n_removed:4d} of {len(tr)} training images removed "
          f"({100*n_removed/len(tr):.1f}%), by class {by_class}")

RESULTS["fold_stats"] = fold_stats
# the test folds MUST be identical across arms - the whole experiment rests on it
for k in range(N_FOLDS):
    a = ARMS["image-level"][k][1]; b = ARMS["dedup"][k][1]; c = ARMS["random-removal"][k][1]
    assert np.array_equal(a, b) and np.array_equal(a, c), "test folds differ between arms"
for k in range(N_FOLDS):
    assert len(ARMS["dedup"][k][0]) == len(ARMS["random-removal"][k][0]), \
        "dedup and random-removal training sets must be the same size"
print("\ntest folds identical across arms: OK")
print("dedup and random-removal training sizes matched: OK")
save_json()

## 4. Models and the run

In [ ]:
SIZE, EPOCHS, BATCH, LR = 224, 30, 16, 1e-4
X = np.empty((len(df), SIZE, SIZE, 3), np.float32)
for i, p in enumerate(tqdm(df["path"], desc=f"load {SIZE}px")):
    X[i] = np.asarray(Image.open(p).convert("RGB").resize((SIZE, SIZE),
                                                          Image.BILINEAR), np.float32)

def _transfer(base_fn, preprocess, size=SIZE, n_classes=3, unfreeze=20):
    inp = layers.Input((size, size, 3))
    x = keras.Sequential([layers.RandomFlip("horizontal"),
                          layers.RandomRotation(0.05),
                          layers.RandomZoom(0.15, 0.15)], name="aug")(inp)
    base = base_fn(include_top=False, weights="imagenet",
                   input_shape=(size, size, 3))
    base.trainable = True
    for layer in base.layers[:-unfreeze]:
        layer.trainable = False
    x = base(preprocess(x), training=False)
    x = layers.Dropout(0.3)(layers.GlobalAveragePooling2D()(x))
    return keras.Model(inp, layers.Dense(n_classes, activation="softmax")(x))

ARCHS = {
    "EfficientNetB0 (pretrained)": lambda: _transfer(
        keras.applications.EfficientNetB0,
        keras.applications.efficientnet.preprocess_input),
    "ResNet50 (pretrained)": lambda: _transfer(
        keras.applications.ResNet50,
        keras.applications.resnet50.preprocess_input),
}

def evaluate(y_true, y_pred):
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=[0, 1, 2], average=None, zero_division=0)
    return {"accuracy": float(accuracy_score(y_true, y_pred)),
            "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
            "macro_f1": float(f1.mean())}

In [ ]:
rows, t0all = [], time.time()
for arch_name, build in ARCHS.items():
    for arm_name, arm_folds in ARMS.items():
        for k, (tr, te) in enumerate(arm_folds):
            keras.backend.clear_session()
            tf.random.set_seed(SEED + k)
            counts = np.bincount(y_all[tr], minlength=3)
            cw = {i: float(len(tr) / (3 * c)) if c else 0.0
                  for i, c in enumerate(counts)}
            model = build()
            model.compile(optimizer=keras.optimizers.Adam(LR),
                          loss="sparse_categorical_crossentropy",
                          metrics=["accuracy"])
            t0 = time.time()
            model.fit(X[tr], y_all[tr], epochs=EPOCHS, batch_size=BATCH,
                      class_weight=cw, verbose=0)
            m = evaluate(y_all[te], model.predict(X[te], verbose=0).argmax(1))
            m.update({"architecture": arch_name, "arm": arm_name, "fold": k,
                      "n_train": int(len(tr)), "n_test": int(len(te)),
                      "minutes": (time.time() - t0) / 60})
            rows.append(m)
            print(f"{arch_name[:14]:14s} | {arm_name:15s} | fold {k} | "
                  f"acc {m['accuracy']:.3f}  bal {m['balanced_accuracy']:.3f}  "
                  f"({m['minutes']:.1f}m)")
            del model; gc.collect()

cv = pd.DataFrame(rows)
cv.to_csv(f"{RESULTS_DIR}/cv_per_fold.csv", index=False)
RESULTS["per_fold"] = cv.to_dict("records")
print(f"\ntotal {(time.time()-t0all)/60:.1f} min over {len(cv)} runs")
save_json()

## 5. The answer

In [ ]:
summary = (cv.groupby(["architecture", "arm"])
             .agg(acc=("accuracy", "mean"), acc_sd=("accuracy", "std"),
                  bal=("balanced_accuracy", "mean"), bal_sd=("balanced_accuracy", "std"),
                  f1=("macro_f1", "mean"), f1_sd=("macro_f1", "std"))
             .round(4).reset_index())
print(summary.to_string(index=False))
summary.to_csv(f"{RESULTS_DIR}/summary.csv", index=False)
RESULTS["summary"] = summary.to_dict("records")

print()
verdict = {}
for arch in cv["architecture"].unique():
    def paired(arm):
        return cv[(cv.architecture == arch) &
                  (cv.arm == arm)].sort_values("fold")["balanced_accuracy"].to_numpy()
    base, dd, rr = paired("image-level"), paired("dedup"), paired("random-removal")
    # paired across folds: the arms share test folds, so per-fold differences are paired
    d_dup = base - dd
    d_rnd = base - rr
    verdict[arch] = {
        "image_level": float(base.mean()),
        "dedup": float(dd.mean()),
        "random_removal": float(rr.mean()),
        "drop_from_dedup": float(d_dup.mean()),
        "drop_from_dedup_sd": float(d_dup.std(ddof=1)),
        "drop_from_random": float(d_rnd.mean()),
        "drop_from_random_sd": float(d_rnd.std(ddof=1)),
        "duplication_effect": float(d_dup.mean() - d_rnd.mean()),
    }
    v = verdict[arch]
    print(f"{arch}")
    print(f"  image-level     {v['image_level']:.4f}")
    print(f"  dedup           {v['dedup']:.4f}   drop {v['drop_from_dedup']:+.4f} "
          f"(sd {v['drop_from_dedup_sd']:.4f})")
    print(f"  random-removal  {v['random_removal']:.4f}   drop {v['drop_from_random']:+.4f} "
          f"(sd {v['drop_from_random_sd']:.4f})")
    print(f"  -> attributable to duplication, size held constant: "
          f"{v['duplication_effect']:+.4f}")
    print()
RESULTS["verdict"] = verdict
save_json()
print("Reading: a drop under dedup that random-removal does NOT reproduce is")
print("direct evidence of near-duplicate leakage. If both drop equally, the")
print("effect is training-set size and the paper's mechanism claim fails.")

In [ ]:
fig, axes = plt.subplots(1, len(ARCHS), figsize=(5.5 * len(ARCHS), 4), squeeze=False)
order = ["image-level", "dedup", "random-removal"]
colours = ["#c0392b", "#2471a3", "#b7950b"]
for ax, arch in zip(axes[0], cv["architecture"].unique()):
    for i, (arm, col) in enumerate(zip(order, colours)):
        v = cv[(cv.architecture == arch) & (cv.arm == arm)]["balanced_accuracy"]
        ax.scatter([i] * len(v), v, color=col, alpha=0.3, s=22, linewidths=0)
        ax.errorbar(i, v.mean(), yerr=v.std(), fmt="o", color=col, markersize=9,
                    capsize=5, linewidth=1.8)
    ax.set_xticks(range(3)); ax.set_xticklabels(order, fontsize=9)
    ax.grid(axis="y", alpha=0.3); ax.set_axisbelow(True)
    ax.set_title(arch, fontsize=10)
axes[0][0].set_ylabel("balanced accuracy")
fig.suptitle("Same test folds. Only the training set differs.", fontsize=11)
fig.tight_layout(rect=(0, 0, 1, 0.93))
fig.savefig(f"{RESULTS_DIR}/fig_dedup_arm.png", dpi=200)
plt.show()

In [ ]:
path = shutil.make_archive(f"{WORK}/fyp_phase2n_results", "zip", RESULTS_DIR)
print("archive:", path, f"({os.path.getsize(path)/1e6:.1f} MB)")
if not IN_KAGGLE:
    try:
        from google.colab import files; files.download(path)
    except Exception as e:
        print("download from the file browser:", e)